In [ ]:
import torch
import numpy as np
import nltk
import evaluate
import math
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

# 引入构建本地数据库和网络搜索所需的库
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document
from duckduckgo_search import DDGS

# 下载 NLTK 句子分割数据包
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# ---------------------------------------------------------
# 1. 环境与配置
# ---------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用计算设备: {device}")

model_checkpoint = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# ---------------------------------------------------------
# 2. 加载并筛选数据集
# ---------------------------------------------------------
print("正在从 Hugging Face 加载数据集...")
dataset_name = "BryanTegomoh/public-health-intelligence-datasetpublic-health-intelligence-dataset"
raw_datasets = load_dataset(dataset_name)

train_dataset = raw_datasets["train"].shuffle(seed=42).select(range(10000))
val_dataset = raw_datasets["validation"].shuffle(seed=42).select(range(1000))

# ---------------------------------------------------------
# 3. 数据预处理
# ---------------------------------------------------------
def preprocess_function(examples):
    inputs = []
    targets = []

    for message_list in examples["messages"]:
        user_text = ""
        assistant_text = ""
        for msg in message_list:
            if msg["role"] == "user":
                user_text = msg["content"]
            elif msg["role"] == "assistant":
                assistant_text = msg["content"]
        inputs.append(user_text)
        targets.append(assistant_text)

    model_inputs = tokenizer(inputs, max_length=512, truncation=True)
    labels = tokenizer(targets, max_length=256, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("正在预处理数据...")
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(preprocess_function, batched=True, remove_columns=val_dataset.column_names)

# ---------------------------------------------------------
# 4. 定义评估指标 (ROUGE Score)
# ---------------------------------------------------------
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]

    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

# ---------------------------------------------------------
# 5. 模型初始化与训练参数
# ---------------------------------------------------------
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to(device)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# =========================================================
# 配置项 1: 正式完整训练配置 (按 Epoch 训练)
# 当需要运行完整的模型微调时，请取消注释这段代码，并注释掉配置项 2。
# =========================================================
# training_args = Seq2SeqTrainingArguments(
#     output_dir="./ph_chatbot_results",
#     eval_strategy="epoch",           # 每完成 1 个完整的数据集循环 (Epoch) 评估一次
#     save_strategy="epoch",
#     learning_rate=3e-5,
#     per_device_train_batch_size=4,   # 若显存不足 (OOM)，可下调至 2 或 1
#     per_device_eval_batch_size=4,
#     weight_decay=0.01,
#     save_total_limit=2,              # 最多保存 2 个模型权重检查点，节省磁盘空间
#     num_train_epochs=3,              # 完整训练 3 轮
#     predict_with_generate=True,      # 关键参数：强制模型在评估时实际生成文本序列，否则无法计算 ROUGE
#     fp16=torch.cuda.is_available(),  # 若有 GPU 则启用半精度计算，加快训练速度并减少显存占用
#     logging_dir='./logs',
# )

# =========================================================
# 配置项 2: 快速测试配置 (当前激活状态)
# 用于快速验证代码流水线是否正确，避免漫长的等待。
# =========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="./ph_chatbot_results",
    eval_strategy="steps",
    eval_steps=10,                   # 每训练 10 步进行一次 ROUGE 评估
    save_steps=10,
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=1,
    max_steps=20,                    # 仅训练 20 步即强制结束。测试通过后请移除此行以使用完整的 num_train_epochs。
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_dir='./logs',
    logging_steps=5,                 # 每 5 步输出一次日志
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer
)

# ---------------------------------------------------------
# 6. 开始训练与验证集评估
# ---------------------------------------------------------
print("\n>>> 开始训练 Chatbot 模型 <<<")
trainer.train()

print("\n>>> 开始在验证集上进行全面评估 (计算 ROUGE Score) <<<")
eval_results = trainer.evaluate()
print("\n最终 ROUGE 评估结果:")
for key, value in eval_results.items():
    print(f"{key}: {value}")


# =========================================================
# 【底层构建 1】本地 PDF 医学数据库 (Local Database)
# =========================================================
print("\n>>> [后台任务] 正在构建本地 WHO 医学数据库... <<<")
local_ensemble_retriever = None

try:
    loader = PyPDFDirectoryLoader("WHO")
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    local_chunks = text_splitter.split_documents(documents)

    bm25_retriever = BM25Retriever.from_documents(local_chunks)
    bm25_retriever.k = 2

    rag_embeddings = HuggingFaceEmbeddings(model_name="pritamdeka/S-PubMedBert-MS-MARCO")
    local_vectorstore = Chroma.from_documents(local_chunks, rag_embeddings, persist_directory="./db_local_who")
    local_vector_retriever = local_vectorstore.as_retriever(search_kwargs={"k": 2})

    local_ensemble_retriever = EnsembleRetriever(retrievers=[bm25_retriever, local_vector_retriever], weights=[0.4, 0.6])
    print(f"✅ 本地医学数据库建立完成 (提取 {len(local_chunks)} 个片段)。")
except Exception as e:
    print(f"⚠️ 本地知识库构建跳过 (未检测到 WHO 文件夹): {e}")


# =========================================================
# 【底层构建 2】实时网络新闻数据库 (Web Database) - Google News RSS 版本
# =========================================================
print("\n>>> [后台任务] 正在抓取近一周网络新闻并构建独立 Web 数据库... <<<")
web_retriever = None
trusted_news_cache = []

try:
    import feedparser
    from html.parser import HTMLParser
    import time
    
    # Google News 的 RSS 订阅源（国际版，聚焦公共卫生）
    GOOGLE_NEWS_URLS = [
        "https://news.google.com/rss/search?q=public+health+policy&hl=en&gl=US&ceid=US:en",
        "https://news.google.com/rss/search?q=disease+outbreak&hl=en&gl=US&ceid=US:en",
        "https://news.google.com/rss/search?q=WHO+health&hl=en&gl=US&ceid=US:en",
    ]
    
    # HTML 标签清理工具
    class MLStripper(HTMLParser):
        def __init__(self):
            super().__init__()
            self.reset()
            self.strict = False
            self.convert_charrefs = True
            self.text = []
        
        def handle_data(self, d):
            self.text.append(d)
        
        def get_data(self):
            return ''.join(self.text)
    
    def strip_html(html):
        stripper = MLStripper()
        try:
            stripper.feed(html)
            return stripper.get_data()
        except:
            return html
    
    web_chunks = []
    
    for feed_url in GOOGLE_NEWS_URLS:
        try:
            print(f"🔍 正在获取订阅源: {feed_url.split('q=')[1].split('&')[0]}...")
            feed = feedparser.parse(feed_url)
            
            if feed.entries:
                for entry in feed.entries[:5]:  # 每个订阅源获取 5 条
                    title = entry.get('title', '')
                    content = entry.get('summary', '')
                    link = entry.get('link', '')
                    
                    # 清理 HTML 标签
                    clean_content = strip_html(content)
                    
                    if title and clean_content and link:
                        doc = Document(
                            page_content=f"Title: {title}. Content: {clean_content}",
                            metadata={
                                'source': link,
                                'published': entry.get('published', '')
                            }
                        )
                        web_chunks.append(doc)
                        trusted_news_cache.append({
                            'title': title,
                            'url': link,
                            'body': clean_content
                        })
                print(f"   ✅ 成功获取 {len([e for e in feed.entries[:5] if e.get('summary')])} 条新闻")
            else:
                print(f"   ⚠️ 该订阅源暂无新闻")
            
            time.sleep(1)  # 避免请求过于频繁
            
        except Exception as feed_error:
            print(f"   ⚠️ 订阅源获取失败: {feed_error}")
    
    if web_chunks:
        print(f"\n📊 总计获取 {len(web_chunks)} 篇新闻，正在构建向量数据库...")
        
        # 确保嵌入模型已初始化
        if 'rag_embeddings' not in locals():
            rag_embeddings = HuggingFaceEmbeddings(model_name="pritamdeka/S-PubMedBert-MS-MARCO")
        
        web_vectorstore = Chroma.from_documents(web_chunks, rag_embeddings, persist_directory="./db_web_news")
        web_retriever = web_vectorstore.as_retriever(search_kwargs={"k": 3})
        print(f"✅ 网络实时数据库建立完成 (已索引 {len(web_chunks)} 篇权威报道)。")
    else:
        print("⚠️ 未成功获取任何新闻。")
        
except Exception as e:
    import traceback
    print(f"⚠️ 网络搜索受限: {e}")
    traceback.print_exc()


# =========================================================
# 【自动执行模块】Application 2: Epidemiological Trend Summarizer
# =========================================================
# 此模块完全独立于 QA，代码运行到此处将自动触发生成。
print("\n" + "="*70)
print("🌍 Application 2: 最近的全球公共卫生要闻简报 (自动生成)")
print("="*70)

if not trusted_news_cache:
    print("❌ 独立网络数据库中没有足够的可信新闻。")
else:
    model.eval()
    # 严格限制只处理并输出前 3 条要闻
    for i, news in enumerate(trusted_news_cache[:3]):
        # 让模型对冗长的新闻 body 进行一句话核心摘要
        prompt = f"Summarize this public health news clearly and concisely in one sentence:\n{news['body']}\nSummary:"
        inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(device)

        with torch.no_grad():
            outputs = model.generate(inputs["input_ids"], max_length=100, num_beams=4, early_stopping=True)
        briefing = tokenizer.decode(outputs[0], skip_special_tokens=True)

        print(f"📰 要闻 {i+1}: {briefing}")
        print(f"   🔗 来源链接: {news['url']}\n")
    print("✅ 简报自动生成完毕！")


# =========================================================
# 【交互模块】Application 1: Public Health QA Chatbot
# =========================================================
# 专属 UI，排在 App 2 下方，完全分离。
import ipywidgets as widgets
from IPython.display import display, clear_output

print("\n" + "="*70)
print("🤖 Application 1: Public Health QA Chatbot (交互问答)")
print("="*70)

qa_input = widgets.Textarea(
    value='What are the latest policies or methods regarding Dengue fever?',
    placeholder='在这里输入公共卫生相关的医学问题...',
    description='提问:',
    layout=widgets.Layout(width='80%', height='60px')
)
qa_button = widgets.Button(description='执行 QA 问答', button_style='primary', icon='comment')
qa_output = widgets.Output()

def run_app1(b):
    with qa_output:
        clear_output(wait=True)
        user_query = qa_input.value
        if not user_query.strip(): return

        print(f"👤 用户提问: {user_query}\n")

        # 分别向两个独立的数据库发起查询
        local_context = ""
        web_context = ""

        if local_ensemble_retriever:
            print("🔍 正在检索本地权威 PDF 数据库...")
            local_docs = local_ensemble_retriever.invoke(user_query)
            local_context = " ".join([doc.page_content for doc in local_docs])

        if web_retriever:
            print("🌐 正在检索独立网络时效数据库...")
            web_docs = web_retriever.invoke(user_query)
            web_context = " ".join([doc.page_content for doc in web_docs])

        augmented_prompt = f"""Use the following dual-source context to answer the question.
[Authoritative Local Context]: {local_context}
[Recent Web News Context]: {web_context}

Question: {user_query}
Answer:"""

        print("🧠 正在融合多源数据生成专业回答...\n")
        inputs = tokenizer(augmented_prompt, return_tensors="pt", max_length=1024, truncation=True).to(device)
        model.eval()
        with torch.no_grad():
            outputs = model.generate(
                inputs["input_ids"], max_length=256, num_beams=4, early_stopping=True,
                return_dict_in_generate=True, output_scores=True
            )

        response = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)

        print(">>> 预测与评估结果 <<<")
        print(f"🤖 Chatbot 回答: {response}\n")

        seq_score = torch.exp(outputs.sequences_scores[0]).item()
        generated_tokens = outputs.sequences[0]
        valid_length = (generated_tokens != tokenizer.pad_token_id).sum().item()
        word_confidence = math.exp(outputs.sequences_scores[0].item() / valid_length)

        print(f"整句总置信度 (Sequence Confidence): {seq_score:.4%}")
        print(f"平均单词置信度 (Avg Word Confidence): {word_confidence:.2%}")

# 绑定事件并单独渲染 App 1 的 UI
qa_button.on_click(run_app1)
display(qa_input, qa_button, qa_output)